In [ ]:
!pip install opendatasets

In [ ]:
from torch.utils.data import Dataset
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from PIL import Image
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import ssl
from IPython.display import clear_output
import pandas as pd
import opendatasets as od
import os
from pathlib import Path
from PIL import Image

dataset_url = 'https://www.kaggle.com/competitions/ml-intensive-yandex-academy-spring-2026/data'
od.download(dataset_url)

#Значения для скачивания данных
#{"username":"spychka","key":"96ba0de1e4c55da9aa22a6e9663cf11f"}


data_path = Path("./ml-intensive-yandex-academy-spring-2026/dataset")
train_images_path = data_path / "train_images"
test_images_path = data_path / "test_images"
csv_path = data_path / "train_solution.csv"


# Загружаем CSV
train_solution = pd.read_csv(Path(csv_path), header=None, names=["id", "label"])
print(train_solution.head())
print(len(train_solution))

#Писал без усреднения, тк у нас картинки, думаю тут это не нужно
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()])


labels_dict = dict(zip(train_solution["id"].astype(str), train_solution["label"]))

class ImageDataset(Dataset):
    #Загружает изображения только при обращении, а не все сразу
    def __init__(self, images_path, transform, max_images=50000, labels_dict=None):
        self.images_path = Path(images_path)
        self.transform = transform
        self.labels_dict = labels_dict

        # Список файлов
        self.image_files = [f for f in self.images_path.iterdir()]

        #Сразу собираем все ответы
        self.all_labels = []
        if self.labels_dict:
          for img_file in self.image_files:
              img_id = img_file.stem
              label = self.labels_dict.get(img_id, -1) if self.labels_dict else -1
              self.all_labels.append(label)

        print(f"Датасет: {len(self.image_files)} изображений")


    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_file = self.image_files[idx]
        img = Image.open(img_file).convert('RGB')
        img_tensor = self.transform(img)

        if self.labels_dict:
          label = self.all_labels[idx]
          return img_tensor, label
        else:
          return img_tensor


# Создаем датасет (изображения не загружаются в память)
train_dataset = ImageDataset(
    train_images_path,
    transform,
    max_images=50000,
    labels_dict=labels_dict
)

test_dataset = ImageDataset(
    test_images_path,
    transform,
    max_images=10000,
    labels_dict=None
)

# DataLoader будет загружать изображения по мере необходимости
# BATCH_SIZE = 32
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
# test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# for image, label in train_loader:
#     print(f"Батч: {image}")
#     print(label)
#     break

Skipping, found downloaded files in "./ml-intensive-yandex-academy-spring-2026" (use force=True to force download)
   id  label
0   0      0
1   1      1
2   2      1
3   3      0
4   4      0
50000
Датасет: 50000 изображений
Датасет: 10000 изображений


In [ ]:
import cv2
from scipy.fft import fft2, fftshift


#Добавляет каналы Лапласа и Собеля
def high_freq_channel(image_tensor):

  img_np = image_tensor.numpy().transpose(1, 2, 0)
  gray = cv2.cvtColor((img_np * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
  gray_float = gray.astype(np.float32) / 255.0

  # Лаплас
  laplacian = cv2.Laplacian(gray_float, cv2.CV_32F, ksize=3)
  laplacian = np.abs(laplacian)
  laplacian = laplacian / (laplacian.max() + 1e-8)

  # Собель
  sobelx = cv2.Sobel(gray_float, cv2.CV_32F, 1, 0, ksize=3)
  sobely = cv2.Sobel(gray_float, cv2.CV_32F, 0, 1, ksize=3)
  sobel = np.sqrt(sobelx**2 + sobely**2)
  sobel = sobel / (sobel.max() + 1e-8)

  return torch.tensor(laplacian).unsqueeze(0), torch.tensor(sobel).unsqueeze(0)


#Добавляет канал FFT спектра
def fft_channel(image_tensor):

    img_np = image_tensor.numpy().transpose(1, 2, 0)
    gray = np.mean(img_np, axis=2)

    f = fft2(gray)
    fshift = fftshift(f)
    magnitude = np.log1p(np.abs(fshift))
    magnitude = (magnitude - magnitude.min()) / (magnitude.max() - magnitude.min() + 1e-8)

    return torch.tensor(magnitude, dtype=torch.float32).unsqueeze(0)

In [ ]:
import random


#Применяет аугментации
def apply_augm(image_tensor, p=0.5):

  img = image_tensor.clone()

  if random.random() < p:
    img = torch.flip(img, dims=[2])

  if random.random() < p:
    angle = random.uniform(-10, 10)
    img = transforms.functional.rotate(img, angle)

  if random.random() < p:
    scale = random.uniform(0.85, 1.15)
    h, w = img.shape[1], img.shape[2]
    new_h, new_w = int(h * scale), int(w * scale)
    img = transforms.functional.resize(img, (new_h, new_w))
    img = transforms.functional.center_crop(img, (h, w))

  # Шум
  if random.random() < p:
    noise = torch.randn_like(img) * 0.05
    img = img + noise

  # Изменение контраста
  if random.random() < p:
    contrast_factor = random.uniform(0.8, 1.2)
    mean = img.mean(dim=[1,2], keepdim=True)
    img = (img - mean) * contrast_factor + mean

  return torch.clamp(img, 0, 1)

In [ ]:
#Датасет с RGB + Лаплас + Собель + FFT = 6 каналов
class New_prod_Dataset(Dataset):
  def __init__(self, og_dataset, use_high_freq=True, use_fft=True, use_augm=False):
    self.original = og_dataset
    self.use_high_freq = use_high_freq
    self.use_fft = use_fft
    self.use_augm = use_augm

    # self.image_files = [f for f in self.images_path.iterdir()]

    #Сразу собираем все ответы
    self.all_labels = []
    if self.use_augm:
      for im, label in self.original:
        self.all_labels.append(label)


  def __len__(self):
    return len(self.original)

  def __getitem__(self, idx):
    img_tensor, label = self.original[idx]

    # Аугментации (только для train)
    if self.use_augm:
      img_tensor = apply_augm(img_tensor)

    # Собираем RGB каналы
    channels = [img_tensor]

    if self.use_high_freq:
      lap, sob = high_freq_channel(img_tensor)
      channels.append(lap)
      channels.append(sob)

    if self.use_fft:
      fft_ch = fft_channel(img_tensor)
      channels.append(fft_ch)

    new_tensor = torch.cat(channels, dim=0)

    return new_tensor, label

In [ ]:
# Для тренировки (с аугментациями)
new_train_dataset = New_prod_Dataset(
    train_dataset,
    use_high_freq=True,
    use_fft=True,
    use_augm=True
)

# Для теста (без аугментаций)
new_test_dataset = New_prod_Dataset(
    test_dataset,
    use_high_freq=True,
    use_fft=True,
    use_augm=False
)

for im, label in new_train_dataset:
  print(im)
  print(label)
  break
#Странно, что так много нулей, буду исправлять по результатам обучения

tensor([[[0.5582, 0.4799, 0.6104,  ..., 0.1427, 0.1235, 0.0288],
         [0.5207, 0.4449, 0.4530,  ..., 0.1507, 0.0000, 0.0207],
         [0.5081, 0.5449, 0.5059,  ..., 0.1765, 0.0112, 0.1048],
         ...,
         [0.8186, 0.9078, 0.8550,  ..., 0.8320, 0.8917, 0.8440],
         [0.8210, 0.8680, 0.9105,  ..., 0.7642, 0.8854, 0.8676],
         [0.8655, 0.8373, 0.9317,  ..., 0.6948, 1.0000, 0.9098]],

        [[0.0128, 0.0000, 0.0042,  ..., 0.0653, 0.0433, 0.0281],
         [0.0156, 0.0072, 0.0000,  ..., 0.1083, 0.0209, 0.0000],
         [0.0000, 0.0783, 0.0000,  ..., 0.1997, 0.0779, 0.0564],
         ...,
         [0.1045, 0.2555, 0.1276,  ..., 0.4030, 0.3488, 0.1364],
         [0.1393, 0.2586, 0.3112,  ..., 0.3642, 0.3270, 0.2023],
         [0.1592, 0.2147, 0.2650,  ..., 0.2191, 0.1183, 0.2724]],

        [[0.0590, 0.0330, 0.0000,  ..., 0.1076, 0.1236, 0.1746],
         [0.0148, 0.0631, 0.0248,  ..., 0.0944, 0.1064, 0.0978],
         [0.0195, 0.0635, 0.0216,  ..., 0.0614, 0.1522, 0.

In [ ]:
from collections import Counter
from torch.utils.data import Subset

#Идём в тупую
#делаю downsampling(делаем одинковове количество классов)
min_samples = min(Counter(new_train_dataset.all_labels).values())
mini_min_samples = min_samples // 10
print(mini_min_samples)

# Получаем индексы
class_0_idx = [i for i, l in enumerate(new_train_dataset.all_labels) if l == 0]
class_1_idx = [i for i, l in enumerate(new_train_dataset.all_labels) if l == 1]

# Выбираем случайные
selected_0 = np.random.choice(class_0_idx, mini_min_samples, replace=False)
selected_1 = np.random.choice(class_1_idx, mini_min_samples, replace=False)


balanced_dataset = Subset(new_train_dataset, list(selected_0) + list(selected_1))
#Финальные названия переменных(можете потом их поменять)
#в train_loader у нас есть тензоры каритнок, а также labels клсаоввые ответы
#на эти картинки (см проход циклом) (те train_solutions нам больше не нужен)
#также подготовлен test_loader

850


# Модель (baseline)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


#Нужно увеличить количество каналлов до 6
class VanillaCNN(nn.Module):
    def __init__(self):
        super(VanillaCNN, self).__init__()
        #5 свёрточных слоёв
        self.conv_layers = nn.Sequential(
            #3-32
            nn.Conv2d(6,32, kernel_size=3, padding=1), # 3канлла -> 6 каналлов
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),#256-128
            #32-64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),#128-64
            #64-128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),#64-32
            #128-256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),#32-16
            #256-512
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),#16-8
        )
         #размер после свёрточных слоёв
        self.fl_size = 512*8*8
        #полносвязная часть
        self.fc_layers = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.fl_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1),
        )
    def forward(self, x):
        #свёрточная часть
        x = self.conv_layers(x)
        #flatten
        x = x.view(x.size(0), -1)
        #полносвязная часть
        x = self.fc_layers(x)
        return x

# Обучение

In [ ]:
device = torch.device("cpu")

# Гиперпараметры
EPOCHS = 10                # попробуем не так много, все-таки baseline
BATCH_SIZE = 32           # уже было, но оставим для ясности
LEARNING_RATE = 0.001
SUBSET_SIZE = 1000        # возьмем только 1000 картинок для быстрого теста


model = VanillaCNN().to(device)

```
neg_count = class_counts[0]   # количество отрицательных примеров (класс 0)
pos_count = class_counts[1]   # количество положительных примеров (класс 1)
pos_weight = torch.tensor([neg_count / pos_count]).to(device) # кол-во отрицательных / кол-во положительных
```
штрафует недостаточно сильно, взяли больше


In [ ]:
class_counts = Counter(train_dataset.all_labels)
print(class_counts)

pos_weight = torch.tensor([7.0]).to(device)

# Функция потерь – взвешенная бинарная кросс-энтропия
# Хотим иметь больше важности у миноритарного класса
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

Counter({0: 41500, 1: 8500})


In [ ]:
from torch.utils.data import Subset
import numpy as np

indices = np.random.choice(len(new_train_dataset), SUBSET_SIZE, replace=False) # train_dataset -> new_train_dataset
small_train_dataset = Subset(new_train_dataset, indices) # train_dataset -> new_train_dataset

# Сделали маленькую подвыборку и на ней работаем
small_train_loader = DataLoader(
    small_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

In [ ]:
"""Эта часть обучения вроде устаревшая

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for images, labels in small_train_loader:
        images, labels = images.to(device), labels.to(device)

        # для BCE (нужны float)
        labels = labels.float().view(-1, 1)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        # логиты в предсказания
        preds = (torch.sigmoid(outputs) > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # Метрики
    avg_loss = total_loss / len(small_train_loader)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    print(epoch+1)
    print(f"loss {avg_loss:.4f}")
    print(f"precision {precision:.4f}")
    print(f"recall {recall:.4f}")
    print(f"f1  {f1:.4f}")
    """

SyntaxError: incomplete input (3870118267.py, line 1)

In [ ]:
# посмотрим на одном батче из теста
"""
model.eval()

with torch.no_grad():
    sample_images, sample_labels = next(iter(small_train_loader))
    sample_images = sample_images.to(device)
    outputs = model(sample_images)
    probs = torch.sigmoid(outputs).cpu().numpy()
    preds = (probs > 0.5).astype(int)

    print("ист метки:", sample_labels.numpy()[:10])
    print("предск", preds.flatten()[:10])
    print("вер", probs.flatten()[:10])
    """

ист метки: [0 0 0 0 0 0 0 1 0 0]
предск [1 0 1 0 1 0 0 1 1 1]
вер [0.6593322  0.09978835 0.61884856 0.35456374 0.56199974 0.42563874
 0.49656078 0.7322943  0.59959275 0.7361061 ]


А че так плохо? Попробуем другое

In [ ]:
device = torch.device("cpu")

# Гиперпараметры
EPOCHS = 20               # попробуем не так много, все-таки baseline
BATCH_SIZE = 32           # уже было, но оставим для ясности
LEARNING_RATE = 0.001
SUBSET_SIZE = 2000        # возьмем 2000 картинок для быстрого теста (не быстрого)


model = VanillaCNN().to(device)

In [ ]:
class_counts = Counter(new_train_dataset.all_labels) # train_dataset -> new_train_dataset
print(class_counts)

pos_weight = torch.tensor([7.0]).to(device)

Counter({0: 41500, 1: 8500})


In [ ]:
import torch
import torch.nn.functional as F

def focal_loss(logits, targets, gamma=2.0, alpha=0.75):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    pt = torch.exp(-bce)
    focal_weight = (1 - pt) ** gamma
    alpha_weight = targets * alpha + (1 - targets) * (1 - alpha)
    loss = alpha_weight * focal_weight * bce
    return loss.mean()

In [ ]:
criterion = lambda outputs, labels: focal_loss(outputs, labels, gamma=2.0, alpha=0.75)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
from torch.utils.data import Subset
import numpy as np

# indices = np.random.choice(len(train_dataset), SUBSET_SIZE, replace=False)
# small_train_dataset = Subset(train_dataset, indices)

small_train_loader = DataLoader(
    balanced_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)
# for im, lbael in small_train_loader:
#   print(im)
#   print(lbael)
#   break

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for images, labels in small_train_loader:
        images, labels = images.to(device), labels.to(device)

        labels = labels.float().view(-1, 1)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)   # вызываем нашу focal_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(small_train_loader)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    print(epoch+1)
    print(f"loss {avg_loss:.4f}")
    print(f"precision {precision:.4f}")
    print(f"recall {recall:.4f}")
    print(f"f1  {f1:.4f}")

1
loss 1.4973
precision 0.5031
recall 0.7682
f1  0.6080
2
loss 0.0769
precision 0.5043
recall 0.9741
f1  0.6645
3
loss 0.0719
precision 0.5051
recall 0.9941
f1  0.6698
4
loss 0.0735
precision 0.5036
recall 0.9965
f1  0.6690
5
loss 0.0733
precision 0.5021
recall 0.9929
f1  0.6669
6
loss 0.0721
precision 0.5018
recall 0.9976
f1  0.6677
7
loss 0.0707
precision 0.5045
recall 0.9965
f1  0.6698
8
loss 0.0715
precision 0.5048
recall 0.9929
f1  0.6693
9
loss 0.0696
precision 0.5057
recall 1.0000
f1  0.6717
10
loss 0.0701
precision 0.5054
recall 0.9906
f1  0.6693
11
loss 0.0706
precision 0.5039
recall 1.0000
f1  0.6701
12
loss 0.0700
precision 0.5030
recall 0.9929
f1  0.6677
13
loss 0.0696
precision 0.5024
recall 0.9976
f1  0.6682
14
loss 0.0687
precision 0.5051
recall 0.9953
f1  0.6701
15
loss 0.0700
precision 0.5076
recall 0.9859
f1  0.6701
16
loss 0.0683
precision 0.5000
recall 1.0000
f1  0.6667
17
loss 0.0688
precision 0.5003
recall 1.0000
f1  0.6669
18
loss 0.0679
precision 0.5000
recall 1

In [ ]:
# посмотрим на одном батче из теста 2
model.eval()

with torch.no_grad():
    sample_images, sample_labels = next(iter(small_train_loader))
    sample_images = sample_images.to(device)
    outputs = model(sample_images)
    probs = torch.sigmoid(outputs).cpu().numpy()
    preds = (probs > 0.5).astype(int)

    print("ист метки:", sample_labels.numpy()[:10])
    print("предск", preds.flatten()[:10])
    print("вер", probs.flatten()[:10])

NameError: name 'model' is not defined